In [1]:
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt

from lightgbm import LGBMRegressor
from utils import add_fractional_year
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import r2_score
from scipy.stats import randint, uniform
from catboost import CatBoostRegressor, Pool
from catboost.utils import get_roc_curve

# Data importing

In [2]:
X_train_1 =pd.read_csv("../data/boost/X_train_boost_part1.csv", index_col = "index")
X_train_2 =pd.read_csv("../data/boost/X_train_boost_part2.csv", index_col = "index")
X_train_3 =pd.read_csv("../data/boost/X_train_boost_part3.csv", index_col = "index")
X_train = pd.concat([X_train_1,X_train_2,X_train_3])

X_test = pd.read_csv("../data/boost/X_test_boost.csv", index_col = "index")

y_train = pd.read_csv("../data/y_train.csv", index_col = "index")
y_test = pd.read_csv("../data/y_test.csv", index_col = "index")


In [3]:
# changing categorical variables into correct type
categorical_cols = ['flat_type', 'town', 'flat_model', 'block', 'street_name' ] 

for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

In [ ]:
# handle date as it is not accepted by lightgbm

# try using year only
X_train_light = add_fractional_year(X_train, date_col='month', new_col='month_fraction')
X_train_light.drop(columns='month', inplace=True) 

X_test_light = add_fractional_year(X_test, date_col='month', new_col='month_fraction')
X_test_light.drop(columns='month', inplace=True) 

,town,flat_type,block,street_name,floor_area_sqm,flat_model,remaining_lease,month_fraction
index,,,,,,,,
545942,TAMPINES,EXECUTIVE,497J,TAMPINES ST 45,139.0,PREMIUM APARTMENT,86.7,2008.25000
444740,KALLANG/WHAMPOA,3 ROOM,463,CRAWFORD LANE,60.0,IMPROVED,76.2,2004.66667
638905,JURONG WEST,5 ROOM,987D,JURONG WEST ST 93,110.0,PREMIUM APARTMENT,93.6,2011.33333
295750,WOODLANDS,5 ROOM,877,WOODLANDS AVE 9,126.0,IMPROVED,94.7,2000.25000
829849,BUKIT MERAH,2 ROOM,28,JLN KLINIK,49.0,STANDARD,47.7,2020.91667
...,...,...,...,...,...,...,...,...
354890,QUEENSTOWN,3 ROOM,5,DOVER CRES,82.0,NEW GENERATION,76.1,2001.83333
903984,PASIR RIS,4 ROOM,190,PASIR RIS ST 12,106.0,MODEL A,69.2,2023.58333
485178,YISHUN,4 ROOM,390,YISHUN AVE 6,104.0,MODEL A,80.9,2006.00000


# LightGBM

In [5]:
# setting parameters
params = {
    'objective': 'regression',           # Type of task: regression
    'metric': 'rmse',                    # Root Mean Squared Error
    'boosting_type': 'gbdt',             # Gradient Boosting Decision Trees
    'learning_rate': 0.1,                # Step size shrinkage
    'num_leaves': 31,                    # Max leaf nodes per tree
    'max_depth': -1,                     # No limit (-1)
    'feature_fraction': 0.9,             # Randomly select 90% of features for each tree
    'bagging_fraction': 0.8,             # Randomly select 80% of data for each iteration
    'bagging_freq': 5,                   # Perform bagging every 5 iterations
    'verbose': 1                      
}

In [6]:
X_train_sub, X_valid, y_train_sub, y_valid = train_test_split(
    X_train_light, y_train, test_size=0.2, random_state=42
)

lgb_train = lgb.Dataset(X_train_sub, label=y_train_sub.values.ravel(), categorical_feature=categorical_cols if categorical_cols else 'auto')
lgb_valid = lgb.Dataset(X_valid, label=y_valid.values.ravel(), reference=lgb_train)

model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=['train', 'valid'],
    num_boost_round=1000
)



[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019586 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3539
[LightGBM] [Info] Number of data points in the train set: 597519, number of used features: 8
[LightGBM] [Info] Start training from score 323716.792710


In [7]:
prediction = model.predict(X_test_light)
r2_score_test = r2_score(y_test, prediction)
print(f"R2 score for LightGBM test = {r2_score_test:.4f}")


R2 score for LightGBM test = 0.9820


## Hyperparameter tuning

In [11]:
model_light = LGBMRegressor(
    objective = 'regression',
    device = 'cpu'
)


# Define parameter space
param_dist = {
    'learning_rate': uniform(0.01, 0.3),
    'num_leaves': randint(20, 3000),
    'max_depth': randint(3, 15),
    'min_child_samples': randint(5, 100),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'reg_alpha': uniform(1e-8, 10),
    'reg_lambda': uniform(1e-8, 10)
}

search = RandomizedSearchCV(
    model_light,
    param_distributions=param_dist,
    n_iter = 50,
    cv = 5,
    scoring='r2',
    verbose = 2,
    n_jobs = -1
)

In [13]:
search.fit(X_train_light, y_train.values.ravel(), 
           eval_set=[(X_valid, y_valid.values.ravel())],
           eval_metric='rmse',
           categorical_feature=categorical_cols)

# Best model
best_model = search.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.022409 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3533
[LightGBM] [Info] Number of data points in the train set: 746899, number of used features: 8
[LightGBM] [Info] Start training from score 323626.670953
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [14]:
# Predict and evaluate
y_pred = best_model.predict(X_test_light)
r2 = r2_score(y_test, y_pred)

print("Best Params:", search.best_params_)
print(f"Test R²: {r2:.4f}")

Best Params: {'colsample_bytree': 0.8470992009988975, 'learning_rate': 0.20671647147554234, 'max_depth': 14, 'min_child_samples': 18, 'num_leaves': 1513, 'reg_alpha': 1.463579017403274, 'reg_lambda': 5.259107353374589, 'subsample': 0.9939773336291013}
Test R²: 0.9831


# Catboost

## Data

In [4]:
X_train['month'] = pd.to_datetime(X_train['month'])
X_test['month'] = pd.to_datetime(X_test['month'])

X_train_sub2, X_valid2, y_train_sub2, y_valid2 = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

In [5]:
print(X_train.dtypes)

month              datetime64[ns]
town                     category
flat_type                category
block                    category
street_name              category
floor_area_sqm            float64
flat_model               category
remaining_lease           float64
dtype: object


In [6]:
# creating pools for catboost

train_pool = Pool(
    data = X_train_sub2,
    label = y_train_sub2.values.ravel(),
    cat_features=categorical_cols,
    timestamp= X_train_sub2['month']
)

valid_pool = Pool(
    data=X_valid2,
    label=y_valid2.values.ravel(),
    cat_features=categorical_cols,
    timestamp= X_valid2['month']
)

test_pool = Pool(
    data=X_test,
    cat_features=categorical_cols,
    timestamp=X_test['month']
)

## Training

In [7]:
model = CatBoostRegressor(
    task_type="GPU",
    iterations=1000,
    learning_rate=0.1,
    depth = 5,
    loss_function='RMSE',
    eval_metric='RMSE',
    verbose = 100,
    early_stopping_rounds=50
)

In [8]:
model.fit(train_pool,eval_set=valid_pool, use_best_model=True)

# Evaluate
y_pred = model.predict(test_pool)
r2 = r2_score(y_test, y_pred)
print(f"CatBoost R² score on test set: {r2:.4f}")

0:	learn: 159728.4106781	test: 160276.4541700	best: 160276.4541700 (0)	total: 56.5ms	remaining: 56.4s
100:	learn: 39270.4915756	test: 38919.3031375	best: 38919.3031375 (100)	total: 3.06s	remaining: 27.3s
200:	learn: 34474.6797234	test: 34026.5571711	best: 34026.5571711 (200)	total: 6.2s	remaining: 24.7s
300:	learn: 32303.1688059	test: 31834.3252171	best: 31834.3252171 (300)	total: 10.1s	remaining: 23.4s
400:	learn: 30992.7195097	test: 30525.7117362	best: 30525.7117362 (400)	total: 13.9s	remaining: 20.8s
500:	learn: 30006.1826503	test: 29558.6594542	best: 29558.6594542 (500)	total: 17.9s	remaining: 17.8s
600:	learn: 29260.1565178	test: 28822.3492253	best: 28822.3492253 (600)	total: 21.6s	remaining: 14.3s
700:	learn: 28687.4866761	test: 28270.4762225	best: 28270.4762225 (700)	total: 25.3s	remaining: 10.8s
800:	learn: 28206.2535445	test: 27827.8291442	best: 27827.8291442 (800)	total: 28.8s	remaining: 7.16s
900:	learn: 27800.6366497	test: 27446.6432452	best: 27446.6432452 (900)	total: 32.8

## Hyperparameter Tuning

In [ ]:
param_grid = {
    'depth': [4, 5, 6],
    'learning_rate': [0.05, 0.1, 0.15],
    'l2_leaf_reg': [3, 5, 7],
    'iterations': [300, 500]
}

model = CatBoostRegressor(task_type="GPU", loss_function='RMSE', random_seed=42)

model.randomized_search(
    param_grid,
    Pool(X_train, y_train.values.ravel(), cat_features=categorical_cols, timestamp=X_train['month']),
    n_iter=10,
    cv=3,
    partition_random_seed=42,
    verbose=True,
    train_size=0.8
)

In [ ]:
search_cb = RandomizedSearchCV(
    model_cb,
    param_distributions=param_dist,
    n_iter=30,
    scoring='r2',
    cv=3,
    verbose=2,
    n_jobs=1
)


In [ ]:
model_cb = CatBoostRegressor(task_type="GPU", loss_function='RMSE', verbose=0, random_state=42)

param_dist = {
    'depth': randint(4, 7),
    'learning_rate': uniform(0.05, 0.2),
    'l2_leaf_reg': uniform(3, 7),
    'iterations': randint(300, 600)
}

model_cb.randomized_search(
    param_dist,
    Pool(X_train, y_train.values.ravel(), cat_features=categorical_cols, timestamp=X_train['month']),
    iterations=10,  # how many parameter sets to try
    cv=3,
    partition_random_seed=42,
    verbose=True,
    calc_cv_statistics=True,
    train_size=0.8
)


# Evaluate tuned model
best_model = search_cb.best_estimator_
y_pred = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print("Best Params:", search_cb.best_params_)
print(f"Tuned CatBoost R²: {r2:.4f}")

Fitting 3 folds for each of 30 candidates, totalling 90 fits


: 

## Visulisation

In [ ]:
importance = best_model.get_feature_importance()
feature_names = best_model.feature_names_

plt.figure(figsize=(10,6))
plt.barh(feature_names, importance)
plt.xlabel("Importance")
plt.title("Feature importance")
plt.show()


In [ ]:
best_model.plot_metrics()